# ACL Tear Detection - Combined Dataset Training (V5 - Fixed)

**Fixes over V4:**
1. **Source-aware slice selection** — uses slices 35–40 for Priyank Saxena data, 14–19 for Kaggle MRI data
2. **CLAHE preprocessing** — harmonizes contrast between the two different MRI sequences
3. **Percentile normalization** — handles intensity outliers before z-score normalization
4. **Clean patient-level majority voting** code

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ============================================================
# Configuration
# ============================================================
DATA_DIR = '/content/drive/MyDrive/dataset/combined'

# Training settings
BATCH_SIZE = 32
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4
NUM_CLASSES = 2  # 0: Normal, 1: Tear (Partial + Complete)
RANDOM_SEED = 42

# Source-aware slice ranges (FIXED — no longer using center slices)
SLICE_RANGE_PRIYANK = (35, 41)   # Slices 35–40 (inclusive) for Priyank Saxena (~50 slices)
SLICE_RANGE_MRI     = (14, 20)   # Slices 14–19 (inclusive) for Kaggle MRI (~32 slices)

# CLAHE settings for contrast harmonization
CLAHE_CLIP_LIMIT = 2.0
CLAHE_TILE_GRID  = (8, 8)

# Percentile normalization settings
PERCENTILE_LOW  = 2
PERCENTILE_HIGH = 98

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from tqdm.notebook import tqdm
import cv2
import warnings
warnings.filterwarnings('ignore')

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# ============================================================
# Load and Explore Metadata
# ============================================================
data_path = Path(DATA_DIR)
metadata = pd.read_csv(data_path / 'metadata.csv')

print(f"Total patients: {len(metadata)}")
print(f"\nOriginal label distribution:")
print(metadata['label_name'].value_counts())

# Convert to binary: 0=Normal, 1=Tear (Partial + Complete)
metadata['label_binary'] = (metadata['label'] > 0).astype(int)
metadata['label_name_binary'] = metadata['label_binary'].map({0: 'Normal', 1: 'Tear'})

print(f"\nBinary label distribution:")
print(metadata['label_name_binary'].value_counts())

# Check class balance
class_counts_df = metadata['label_binary'].value_counts().sort_index()
imbalance_ratio = class_counts_df.max() / class_counts_df.min()
print(f"\nClass imbalance ratio: {imbalance_ratio:.1f}x")

# Show source distribution
if 'source' in metadata.columns:
    print(f"\nSource distribution:")
    print(metadata['source'].value_counts())
    print(f"\nSlice counts by source:")
    print(metadata.groupby('source')['num_slices'].describe())

In [ ]:
# ============================================================
# Verify Slice Ranges — Visualize what OLD vs NEW selects
# ============================================================
print("SLICE SELECTION COMPARISON (Old vs New):")
print("=" * 70)

for source_name in metadata['source'].unique():
    source_data = metadata[metadata['source'] == source_name].head(3)
    print(f"\n--- Source: {source_name} ---")
    
    for _, row in source_data.iterrows():
        n = int(row['num_slices'])
        
        # Old method: center 15
        old_start = max(0, (n - 15) // 2)
        old_end = min(n, old_start + 15)
        
        # New method: source-aware
        if source_name == 'Priyank_Saxena':
            new_start, new_end = SLICE_RANGE_PRIYANK
        else:
            new_start, new_end = SLICE_RANGE_MRI
        new_start = min(new_start, n)
        new_end = min(new_end, n)
        
        print(f"  {row['filename']}: {n} slices")
        print(f"    OLD: slices [{old_start}-{old_end}) = {old_end-old_start} slices")
        print(f"    NEW: slices [{new_start}-{new_end}) = {new_end-new_start} slices  ✅")

In [ ]:
# ============================================================
# Train/Val/Test Split (Patient-level to avoid data leakage)
# ============================================================
train_df, temp_df = train_test_split(
    metadata, test_size=0.3, stratify=metadata['label_binary'], random_state=RANDOM_SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label_binary'], random_state=RANDOM_SEED
)

print(f"Train: {len(train_df)} patients")
print(f"Val:   {len(val_df)} patients")
print(f"Test:  {len(test_df)} patients")

print(f"\nTrain label distribution (binary):")
print(train_df['label_name_binary'].value_counts())

# Check source balance in each split
for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f"\n{name} source distribution:")
    print(df['source'].value_counts())

In [ ]:
# ============================================================
# Dataset Class — FIXED with source-aware slicing + CLAHE
# ============================================================

def apply_clahe(img, clip_limit=CLAHE_CLIP_LIMIT, tile_grid=CLAHE_TILE_GRID):
    """
    Apply CLAHE to normalize contrast across different MRI sequences.
    Harmonizes the brightness difference between Priyank Saxena (brighter)
    and Kaggle MRI (darker, more contrasty) data.
    """
    img_uint8 = (img * 255).astype(np.uint8) if img.max() <= 1.0 else img.astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    return clahe.apply(img_uint8).astype(np.float32) / 255.0


def percentile_normalize(img, low=PERCENTILE_LOW, high=PERCENTILE_HIGH):
    """
    Clip pixel values to percentile range, then normalize to 0-1.
    Handles intensity outliers better than simple min-max.
    """
    p_low, p_high = np.percentile(img, [low, high])
    img = np.clip(img, p_low, p_high)
    denom = p_high - p_low
    if denom > 0:
        img = (img - p_low) / denom
    return img


class ACLSliceDataset(Dataset):
    """
    Dataset with source-aware slice selection and CLAHE contrast harmonization.
    
    Key fixes:
    - Uses explicit slice ranges per source instead of geometric center
    - Applies CLAHE to standardize contrast between different MRI sequences
    - Uses percentile normalization before z-score to handle outliers
    """
    def __init__(self, df, data_dir, transform=None, 
                 slice_range_priyank=SLICE_RANGE_PRIYANK, 
                 slice_range_mri=SLICE_RANGE_MRI,
                 use_clahe=True):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.transform = transform
        self.slice_range_priyank = slice_range_priyank
        self.slice_range_mri = slice_range_mri
        self.use_clahe = use_clahe

        # Pre-compute slice indices with SOURCE-AWARE selection
        self.samples = []
        skipped = 0
        for idx, row in self.df.iterrows():
            num_slices = int(row['num_slices'])
            source = row.get('source', 'unknown')
            
            # Select slice range based on data source
            if source == 'Priyank_Saxena':
                start, end = self.slice_range_priyank
            else:  # MRI (Kaggle)
                start, end = self.slice_range_mri
            
            # Clamp to valid range
            start = min(start, num_slices)
            end = min(end, num_slices)
            
            if start >= end:
                skipped += 1
                continue
            
            for slice_idx in range(start, end):
                self.samples.append((idx, slice_idx, int(row['label_binary'])))
        
        if skipped > 0:
            print(f"  ⚠️ Skipped {skipped} patients (slice range out of bounds)")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        patient_idx, slice_idx, label = self.samples[idx]
        row = self.df.iloc[patient_idx]

        # Load volume
        file_path = self.data_dir / row['filename']
        volume = np.load(file_path)['data']

        # Clamp slice_idx to valid range
        slice_idx = min(slice_idx, volume.shape[0] - 1)

        # Get slice as float32 [0, 1]
        img = volume[slice_idx].astype(np.float32) / 255.0

        # Step 1: Percentile normalization (clip outliers)
        img = percentile_normalize(img)

        # Step 2: CLAHE contrast harmonization
        if self.use_clahe:
            img = apply_clahe(img)

        # Step 3: Per-image z-score normalization
        img_mean = img.mean()
        img_std = img.std() + 1e-8
        img = (img - img_mean) / img_std

        # Convert to 3-channel for pretrained models
        img = np.stack([img, img, img], axis=0)
        img = torch.from_numpy(img)

        if self.transform:
            img = self.transform(img)

        return img, label, patient_idx

    def get_labels(self):
        return [s[2] for s in self.samples]

In [ ]:
# ============================================================
# Visualize Preprocessing Pipeline — Before vs After
# ============================================================
fig, axes = plt.subplots(2, 4, figsize=(18, 8))

# Pick one sample from each source
for row_idx, source_name in enumerate(['Priyank_Saxena', 'MRI']):
    sample_row = metadata[metadata['source'] == source_name].iloc[0]
    vol = np.load(data_path / sample_row['filename'])['data']
    
    if source_name == 'Priyank_Saxena':
        s = SLICE_RANGE_PRIYANK[0] + 2  # Middle of ACL range
    else:
        s = SLICE_RANGE_MRI[0] + 2
    
    raw = vol[s].astype(np.float32) / 255.0
    
    # Step-by-step
    step1 = percentile_normalize(raw.copy())
    step2 = apply_clahe(step1.copy())
    step3 = (step2 - step2.mean()) / (step2.std() + 1e-8)
    
    titles = ['Raw', '+ Percentile Norm', '+ CLAHE', '+ Z-Score']
    images = [raw, step1, step2, step3]
    
    for col_idx, (title, img) in enumerate(zip(titles, images)):
        ax = axes[row_idx, col_idx]
        ax.imshow(img, cmap='gray')
        ax.set_title(f'{title}\n(μ={img.mean():.2f}, σ={img.std():.2f})', fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        if col_idx == 0:
            ax.set_ylabel(source_name, fontsize=12, fontweight='bold')

plt.suptitle('Preprocessing Pipeline Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Create Datasets and DataLoaders
# ============================================================

# Data augmentation for training
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
])

val_transform = None

# Create datasets with source-aware slicing
print("Creating datasets with source-aware slice selection...")
print(f"  Priyank Saxena: slices {SLICE_RANGE_PRIYANK[0]}–{SLICE_RANGE_PRIYANK[1]-1}")
print(f"  Kaggle MRI:     slices {SLICE_RANGE_MRI[0]}–{SLICE_RANGE_MRI[1]-1}")
print()

print("Train:")
train_dataset = ACLSliceDataset(train_df, DATA_DIR, train_transform)
print("Val:")
val_dataset = ACLSliceDataset(val_df, DATA_DIR, val_transform)
print("Test:")
test_dataset = ACLSliceDataset(test_df, DATA_DIR, val_transform)

print(f"\nTrain samples (slices): {len(train_dataset)}")
print(f"Val samples (slices):   {len(val_dataset)}")
print(f"Test samples (slices):  {len(test_dataset)}")

In [ ]:
# ============================================================
# Weighted Sampler for Balanced Training
# ============================================================
train_labels = train_dataset.get_labels()
class_counts = np.bincount(train_labels)
class_weights = 1.0 / class_counts
sample_weights = [class_weights[label] for label in train_labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

print(f"Class counts in training: {class_counts}")
print(f"Class weights: {class_weights}")

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
# ============================================================
# Model: EfficientNet-B0 with more trainable layers
# ============================================================
model = models.efficientnet_b0(weights='IMAGENET1K_V1')

# Freeze only first 50% of layers (more adaptation for MRI)
all_params = list(model.features.parameters())
freeze_until = len(all_params) // 2
for i, param in enumerate(all_params):
    param.requires_grad = (i >= freeze_until)

# Replace classifier
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(model.classifier[1].in_features, 256),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(256, NUM_CLASSES)
)

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

In [ ]:
# ============================================================
# Loss, Optimizer, Scheduler
# ============================================================
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)

In [ ]:
# ============================================================
# Training & Validation Functions
# ============================================================

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, (images, labels, _) in enumerate(tqdm(loader, desc='Training')):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return total_loss / len(loader), 100. * correct / total


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels, _ in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / len(loader), 100. * correct / total, all_preds, all_labels

In [ ]:
# ============================================================
# Training Loop with Early Stopping
# ============================================================
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0
patience = 10
patience_counter = 0

SAVE_PATH = '/content/drive/MyDrive/dataset/best_acl_model_v5_fixed.pth'

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_preds, val_labels = validate(model, val_loader, criterion, device)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  -> Saved best model (val_acc: {val_acc:.2f}%)")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

In [ ]:
# ============================================================
# Plot Training History
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history['train_acc'], label='Train Acc')
axes[1].plot(history['val_acc'], label='Val Acc')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/training_history_v5.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# Load Best Model & Evaluate on Test Set
# ============================================================
model.load_state_dict(torch.load(SAVE_PATH))

test_loss, test_acc, test_preds, test_labels = validate(model, test_loader, criterion, device)
print(f"\nTest Results:")
print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.2f}%")

In [ ]:
# ============================================================
# Slice-Level Classification Report
# ============================================================
label_names = ['Normal', 'Tear']
print("\nSlice-level Classification Report:")
print(classification_report(test_labels, test_preds, target_names=label_names, digits=3))

In [ ]:
# ============================================================
# Confusion Matrix — Slice Level
# ============================================================
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Slice Level')
plt.savefig('/content/drive/MyDrive/dataset/confusion_matrix_v5_slice.png', dpi=150)
plt.show()

# Per-class accuracy (slice-level)
print("\nPer-class accuracy (slice-level):")
for i, name in enumerate(label_names):
    class_mask = np.array(test_labels) == i
    if class_mask.sum() > 0:
        class_acc = (np.array(test_preds)[class_mask] == i).mean() * 100
        print(f"  {name}: {class_acc:.1f}%")

In [ ]:
# ============================================================
# Patient-Level Majority Voting
# ============================================================
print("=" * 50)
print("PATIENT-LEVEL RESULTS (Majority Voting)")
print("=" * 50)

# Get patient indices from test dataset
test_patient_indices = [test_dataset.samples[i][0] for i in range(len(test_dataset))]

# Group predictions by patient
patient_preds_dict = {}
patient_labels_dict = {}

for pred, label, patient_idx in zip(test_preds, test_labels, test_patient_indices):
    if patient_idx not in patient_preds_dict:
        patient_preds_dict[patient_idx] = []
        patient_labels_dict[patient_idx] = label
    patient_preds_dict[patient_idx].append(pred)

# Majority voting per patient
patient_final_preds = []
patient_final_labels = []

for patient_idx in sorted(patient_preds_dict.keys()):
    preds = patient_preds_dict[patient_idx]
    final_pred = Counter(preds).most_common(1)[0][0]
    patient_final_preds.append(final_pred)
    patient_final_labels.append(patient_labels_dict[patient_idx])

# Patient-level accuracy
patient_acc = np.mean(np.array(patient_final_preds) == np.array(patient_final_labels)) * 100
print(f"\nTotal test patients: {len(patient_final_preds)}")
print(f"Patient-level Accuracy: {patient_acc:.1f}%")

# Patient-level classification report
print("\nPatient-level Classification Report:")
print(classification_report(patient_final_labels, patient_final_preds, 
                            target_names=label_names, digits=3))

# Patient-level confusion matrix
cm_patient = confusion_matrix(patient_final_labels, patient_final_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_patient, annot=True, fmt='d', cmap='Greens',
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Patient Level (Majority Voting)')
plt.savefig('/content/drive/MyDrive/dataset/confusion_matrix_v5_patient.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# Per-Source Accuracy Analysis
# ============================================================
print("\n" + "=" * 50)
print("PER-SOURCE ACCURACY BREAKDOWN")
print("=" * 50)

# Map patient indices back to source
for source_name in test_df['source'].unique():
    source_patients = test_df[test_df['source'] == source_name].index.tolist()
    # Get patient indices that are from this source
    source_patient_indices = []
    for i, row in test_dataset.df.iterrows():
        if row['source'] == source_name:
            source_patient_indices.append(i)
    
    source_preds = []
    source_labels = []
    for pidx in source_patient_indices:
        if pidx in patient_preds_dict:
            preds = patient_preds_dict[pidx]
            source_preds.append(Counter(preds).most_common(1)[0][0])
            source_labels.append(patient_labels_dict[pidx])
    
    if source_preds:
        acc = np.mean(np.array(source_preds) == np.array(source_labels)) * 100
        print(f"\n{source_name}: {acc:.1f}% ({len(source_preds)} patients)")
        print(classification_report(source_labels, source_preds,
                                    target_names=label_names, digits=3))